# Decision Trees

**Companion lesson:** https://ml-viz.vercel.app/courses/knn-decision-trees/02-decision-trees

This notebook implements the same math as the lesson: Gini and entropy impurity,
information gain for a single split, a full scan to find the best split on a tiny
loan-approval dataset, and how tree depth controls overfitting. Every number here
matches the worked examples in the lesson.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Dark matplotlib style (matches the site theme)
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## The running dataset

One feature (age) and a binary label (1 = approved, 0 = denied). 5 approvals,
5 denials, with a little noise: a 33-year-old approved, a 40-year-old denied.

In [ ]:
ages   = np.array([22, 25, 28, 30, 33, 36, 40, 45, 50, 60])
labels = np.array([ 0,  0,  0,  0,  1,  1,  0,  1,  1,  1])

print('approvals (1):', int(labels.sum()))
print('denials   (0):', int((labels == 0).sum()))

## Impurity measures

$$\text{Gini}(S) = 1 - \sum_k p_k^2 \qquad \text{Entropy}(S) = -\sum_k p_k \log_2 p_k$$

At the root (5/5) we expect Gini = 0.500 and Entropy = 1.000.

In [ ]:
def gini(y):
    _, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return 1 - np.sum(probs ** 2)

def entropy(y):
    _, counts = np.unique(y, return_counts=True)
    probs = counts / counts.sum()
    return -np.sum(probs * np.log2(probs + 1e-10))

print('root Gini    = {:.3f}'.format(gini(labels)))
print('root Entropy = {:.3f}'.format(entropy(labels)))

## Plot impurity vs class balance

Both measures peak at a 50/50 mix and fall to 0 when a node is pure.

In [ ]:
p = np.linspace(0, 1, 200)
gini_curve = 1 - p**2 - (1 - p)**2
ent_curve = -(p * np.log2(p + 1e-10) + (1 - p) * np.log2(1 - p + 1e-10))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(p, gini_curve, color='#818cf8', linewidth=2, label='Gini')
ax.plot(p, ent_curve, color='#14b8a6', linewidth=2, label='Entropy')
ax.axvline(0.5, color='#94a3b8', linestyle='--', linewidth=1)
ax.set_title('Impurity vs class balance', color='white')
ax.set_xlabel('p(class = 1)')
ax.set_ylabel('impurity')
ax.legend()
plt.tight_layout()
plt.show()

print('Gini at 50/50:    {:.3f}'.format(1 - 0.5**2 - 0.5**2))
print('Entropy at 50/50: {:.3f}'.format(-(0.5*np.log2(0.5) + 0.5*np.log2(0.5))))

## Information gain for one split

$$\text{Gain} = I(\text{parent}) - \Big( \tfrac{n_L}{n} I(L) + \tfrac{n_R}{n} I(R) \Big)$$

We score the split **Age <= 31.5**. Expect: left pure (Gini 0), right 5/1
(Gini 0.278), weighted Gini 0.167, Gini gain 0.333.

In [ ]:
def split_score(ages, y, threshold, impurity=gini):
    left_mask = ages <= threshold
    right_mask = ~left_mask
    n = len(y)
    nL, nR = left_mask.sum(), right_mask.sum()
    iL = impurity(y[left_mask])
    iR = impurity(y[right_mask])
    weighted = (nL / n) * iL + (nR / n) * iR
    gain = impurity(y) - weighted
    return iL, iR, weighted, gain

iL, iR, weighted, gain = split_score(ages, labels, 31.5, impurity=gini)
print('Age <= 31.5')
print('  left  Gini = {:.4f}'.format(iL))
print('  right Gini = {:.4f}'.format(iR))
print('  weighted   = {:.4f}'.format(weighted))
print('  Gini gain  = {:.4f}'.format(gain))

_, _, _, ig = split_score(ages, labels, 31.5, impurity=entropy)
print('  info gain (entropy) = {:.4f}'.format(ig))

## Scan every threshold to find the best split

Candidate thresholds are the midpoints between consecutive sorted ages. The
winner should be Age <= 31.5 with Gini gain 0.3333.

In [ ]:
order = np.argsort(ages)
sorted_ages = ages[order]
thresholds = (sorted_ages[:-1] + sorted_ages[1:]) / 2

best_gain, best_thresh = -1.0, None
print('{:>6}  {:>8}  {:>9}'.format('t', 'wGini', 'gain'))
for t in thresholds:
    _, _, weighted, gain = split_score(ages, labels, t, impurity=gini)
    marker = '  <-- best so far' if gain > best_gain else ''
    if gain > best_gain:
        best_gain, best_thresh = gain, t
    print('{:>6.1f}  {:>8.4f}  {:>9.4f}{}'.format(t, weighted, gain, marker))

print()
print('BEST: Age <= {:.1f}  (Gini gain = {:.4f})'.format(best_thresh, best_gain))

In [ ]:
# Visualize the gain for every candidate threshold
gains = [split_score(ages, labels, t, impurity=gini)[3] for t in thresholds]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, gains, 'o-', color='#818cf8', linewidth=2)
ax.axvline(best_thresh, color='#14b8a6', linestyle='--', linewidth=1,
           label='best = {:.1f}'.format(best_thresh))
ax.set_title('Information gain vs split threshold', color='white')
ax.set_xlabel('age threshold')
ax.set_ylabel('Gini gain')
ax.legend()
plt.tight_layout()
plt.show()

## Tree depth controls overfitting

On a noisier 2D dataset, an unconstrained tree memorizes the training set
(train accuracy ~1.0) but generalizes worse. Cross-validation reveals the
sweet spot.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_moons
from sklearn.model_selection import cross_val_score

Xm, ym = make_moons(n_samples=300, noise=0.3, random_state=0)
for depth in [1, 3, 5, None]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=0)
    cv = cross_val_score(clf, Xm, ym, cv=5).mean()
    clf.fit(Xm, ym)
    train = clf.score(Xm, ym)
    print('max_depth={:>4}: train = {:.3f}   CV = {:.3f}'.format(str(depth), train, cv))

## Key takeaways

- Trees split greedily on the feature/threshold that most reduces **impurity** (Gini or entropy).
- **Information gain** = parent impurity - weighted child impurity.
- Unconstrained trees overfit (train accuracy ~1.0, lower CV); limit `max_depth` / `min_samples_leaf` or prune.
- Trees are interpretable and need no feature scaling, but are high-variance alone - hence ensembles.

---
## ✏️ Your turn

Exercise scaffolds: concept recapped, outline set, `# TODO(you)` marks your part. The `assert` cell tells you when you've got it.

### Exercise 1 — Gini impurity

The Gini impurity of a label set is $G = 1 - \sum_c p_c^2$, where $p_c$ is the fraction of samples in class $c$. It is 0 for a pure node and maximal (0.5 for two classes) for a 50/50 split.

In [ ]:
def gini(labels):
    """Gini impurity of a 1-D array of class labels."""
    labels = np.asarray(labels)
    if labels.size == 0:
        return 0.0

    # TODO(you): class probabilities (hint: np.unique with return_counts=True)
    probs = ...

    # TODO(you): 1 - sum of squared probabilities
    return ...

In [ ]:
# Checks — run me
assert gini([1, 1, 1, 1]) == 0.0, "pure node has zero impurity"
assert abs(gini([0, 0, 1, 1]) - 0.5) < 1e-12, "50/50 two-class split -> 0.5"
assert abs(gini([0, 0, 0, 1]) - 0.375) < 1e-12, "3:1 split -> 1 - (9/16 + 1/16)"
assert abs(gini([0, 1, 2]) - 2/3) < 1e-12, "three equal classes -> 2/3"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def gini(labels):
    labels = np.asarray(labels)
    if labels.size == 0:
        return 0.0
    _, counts = np.unique(labels, return_counts=True)
    probs = counts / labels.size
    return float(1 - np.sum(probs ** 2))
```

</details>

### Exercise 2 — Impurity decrease of a split

A split's quality is the **weighted impurity decrease**:

$\Delta = G(\text{parent}) - \frac{n_L}{n} G(\text{left}) - \frac{n_R}{n} G(\text{right})$

The best split maximizes $\Delta$. Implement it, then verify that a perfect split of a 50/50 parent gains the full 0.5.

In [ ]:
def split_gain(parent_labels, left_labels, right_labels):
    """Weighted Gini decrease of splitting parent into (left, right)."""
    parent = np.asarray(parent_labels)
    left = np.asarray(left_labels)
    right = np.asarray(right_labels)
    n = parent.size

    # TODO(you): apply the formula above using your gini()
    return ...

In [ ]:
# Checks — run me
parent = [0, 0, 0, 1, 1, 1]
assert abs(split_gain(parent, [0, 0, 0], [1, 1, 1]) - 0.5) < 1e-12, "perfect split gains full 0.5"
assert abs(split_gain(parent, [0, 0, 1], [0, 1, 1]) - (0.5 - 4/9)) < 1e-12
assert split_gain(parent, parent, []) < 1e-12, "useless split gains nothing"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def split_gain(parent_labels, left_labels, right_labels):
    parent = np.asarray(parent_labels)
    left = np.asarray(left_labels)
    right = np.asarray(right_labels)
    n = parent.size
    return float(gini(parent)
                 - left.size / n * gini(left)
                 - right.size / n * gini(right))
```

</details>